# CoT Trace Generation — Variant D

Generates ~2,000 Chain-of-Thought reasoning traces from e-SNLI training examples  
using **DeepSeek-R1-Distill-Qwen-7B** via the HuggingFace Inference API.

**Target:** 667 traces per label × 3 labels = 2,001 total (balanced)

**Rate limits:** HF free tier allows ~500–1,000 requests/day.  
Run this notebook daily — it auto-resumes from where it left off each session.

---

**Before running:**
1. Get a free HF token at https://huggingface.co/settings/tokens
2. Paste it in Cell 2 below
3. Set `PROJECT_ROOT` to your Drive path in Cell 3
4. Run all cells — generation starts automatically

In [ ]:
# Cell 1 — Install dependencies (run once)
!pip install -q requests pandas

In [ ]:
# Cell 2 — CONFIGURATION (only cell you need to edit)
# ─────────────────────────────────────────────────────────────────

# Your HuggingFace API token — get one at https://huggingface.co/settings/tokens
HF_TOKEN = "hf_PASTE_YOUR_TOKEN_HERE"

# Model to use — best open-source reasoning model on HF free tier
MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"

# Number of traces per label (667 × 3 = 2,001 total)
# Reduce to 333 if you want ~1K total instead
N_PER_LABEL = 667

# ─────────────────────────────────────────────────────────────────

assert HF_TOKEN != "hf_PASTE_YOUR_TOKEN_HERE", \
    "Please paste your HuggingFace token in HF_TOKEN above."

print(f"Model:         {MODEL_ID}")
print(f"Target traces: {N_PER_LABEL * 3:,} ({N_PER_LABEL} per label)")
print("Token:         " + HF_TOKEN[:8] + "..." + HF_TOKEN[-4:])

In [ ]:
# Cell 3 — Mount Google Drive and set project path
from google.colab import drive
drive.mount("/content/drive")

import os, sys

# Set this to your project folder on Google Drive
PROJECT_ROOT = "/content/drive/MyDrive/Old-Explanations-New-Rationales-Bridging-e-SNLI-and-LLM-Chain-of-Thought-in-Encoder-Based-NLI"

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

OUTPUT_PATH = os.path.join(PROJECT_ROOT, "Datasets", "CoT", "cot_traces.csv")

print("Working directory:", os.getcwd())
print("Output CSV will be saved to:", OUTPUT_PATH)

In [ ]:
# Cell 4 — Check existing progress
import pandas as pd
from pathlib import Path

output_path = Path(OUTPUT_PATH)

if output_path.exists():
    df_existing = pd.read_csv(output_path)
    done = df_existing["cot_rationale"].notna() & (df_existing["cot_rationale"].str.strip() != "")
    print(f"Existing file found: {output_path}")
    print(f"  Total rows:      {len(df_existing):,}")
    print(f"  Completed rows:  {done.sum():,}")
    print(f"  Remaining:       {N_PER_LABEL * 3 - done.sum():,}")
    print()
    print("Label breakdown so far:")
    print(df_existing.loc[done, "gold_label"].value_counts().to_string())
else:
    print("No existing file — starting fresh.")
    print(f"Will generate {N_PER_LABEL * 3:,} traces and save to:")
    print(f"  {OUTPUT_PATH}")

In [ ]:
# Cell 5 — Load e-SNLI data and select subset
from data.cot_generation import _load_esnli_train, select_subset

print("Loading e-SNLI training data...")
esnli_df = _load_esnli_train(Path(PROJECT_ROOT))
print(f"  Loaded {len(esnli_df):,} examples")

print("Selecting stratified subset...")
subset = select_subset(esnli_df, n_per_label=N_PER_LABEL)
print(f"  Subset size: {len(subset):,}")
print(f"  Label counts:")
print(subset["gold_label"].value_counts().to_string())
print()
print("Sample row:")
row = subset.iloc[0]
print(f"  Premise:     {row['Sentence1']}")
print(f"  Hypothesis:  {row['Sentence2']}")
print(f"  Label:       {row['gold_label']}")

In [ ]:
# Cell 6 — Quick API test (generates ONE trace to confirm token + model work)
from data.cot_generation import call_hf_inference, _build_prompt

test_prompt = _build_prompt(
    premise="A dog is running through a field.",
    hypothesis="An animal is outside.",
    label="entailment",
)

print("Testing HF API...")
test_result = call_hf_inference(test_prompt, hf_token=HF_TOKEN, model=MODEL_ID)

if test_result:
    print("✓ API working. Sample output:")
    print("-" * 50)
    print(test_result)
    print("-" * 50)
else:
    print("✗ API call failed. Check your token and model name.")
    print("  Token:  ", HF_TOKEN[:8] + "...")
    print("  Model:  ", MODEL_ID)

In [ ]:
# Cell 7 — Generate (or resume) CoT traces
#
# This cell runs until the daily rate limit is hit, then stops.
# Run the notebook again tomorrow — it will resume automatically.
# Progress is saved after EVERY example.

from data.cot_generation import generate_cot_traces

print("Starting generation...")
print("(Will pause automatically when rate limit is hit — just re-run tomorrow)")
print()

generate_cot_traces(
    source_df=subset,
    output_path=OUTPUT_PATH,
    hf_token=HF_TOKEN,
    model=MODEL_ID,
    sleep_between=1.5,   # 1.5s between requests to stay within rate limits
)

In [ ]:
# Cell 8 — Validate quality of completed (or partial) traces
from data.cot_generation import validate_traces

validate_traces(OUTPUT_PATH)

In [ ]:
# Cell 9 — Preview sample traces (optional, inspect quality manually)
import pandas as pd

df = pd.read_csv(OUTPUT_PATH)
done = df["cot_rationale"].notna() & (df["cot_rationale"].str.strip() != "")
sample = df[done].groupby("gold_label").head(1).reset_index(drop=True)

for _, row in sample.iterrows():
    print(f"Label:       {row['gold_label']}")
    print(f"Premise:     {row['Sentence1']}")
    print(f"Hypothesis:  {row['Sentence2']}")
    print(f"Human expl:  {row['Explanation_1']}")
    print(f"CoT trace:")
    print(row['cot_rationale'])
    print("-" * 60)